# Introducción a la Ciencia de Datos: Tarea 1

Este notebook contiene el código de base para realizar la Tarea 1 del curso. Puede copiarlo en su propio repositorio y trabajar sobre el mismo.
Las **instrucciones para ejecutar el notebook** están en la [página inicial del repositorio](https://gitlab.fing.edu.uy/maestria-cdaa/intro-cd).

Se utiliza el lenguaje Python y la librería Pandas. Si no tiene ninguna familiaridad con la librería, se recomienda realizar algún tutorial introductorio (ver debajo).
También se espera que los alumnos sean proactivos a la hora de consultar las documentaciones de las librerías y del lenguaje, para entender el código provisto.
Además de los recursos provistos en la [página del curso](https://eva.fing.edu.uy/course/view.php?id=1378&section=1), los siguientes recursos le pueden resultar interesantes:
 - [Pandas getting started](https://pandas.pydata.org/docs/getting_started/index.html#getting-started) y [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html): Son parte de la documentación en la página oficial de Pandas.
 - [Kaggle Learn](https://www.kaggle.com/learn): Incluye tutoriales de Python y Pandas.


Si desea utilizar el lenguaje R y está dispuesto a no utilizar (o traducir) este código de base, también puede hacerlo.

En cualquier caso, **se espera que no sea necesario revisar el código para corregir la tarea**, ya que todos los resultados y análisis relevantes deberían estar en el **informe en formato PDF**.

## Cargar bibliotecas (dependencias)
Recuerde instalar los requerimientos (`requirements.txt`) en el mismo entorno donde está ejecutando este notebook (ver [README]).

In [ ]:
## ceamos el venv x las  dudas
!test -d .venv ||  py -3.13 -m venv introcd .venv/ 

In [ ]:
## en windows
!if not exist .venv py -3.11 -m venv introcd .venv
!python -m pip install --upgrade pip setuptools wheel
!pip  install -r .\requirements.txt
!pip install -U setuptools

In [ ]:
##CCheckeo  que ande el pofile  de ydaata-profiling

import sys
!{sys.executable} -m pip install --force-reinstall "setuptools<81"
import pkg_resources
print(pkg_resources)
from ydata_profiling import ProfileReport

In [ ]:
from time import time
from pathlib import Path
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import numpy as np
from datasets import load_dataset
from ydata_profiling import ProfileReport
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    
# Agregue aqui el resto de las librerias que necesite
# from ...
# import ...

## Descarga del dataset
En esta tarea se utilizará una base de datos abierta que contiene artículos de noticias publicados en distintos medios de prensa, con la finalidad de realizar una clasificación de textos según el medio de prensa al que pertenecen. [Link](https://huggingface.co/datasets/rjac/all-the-news-2-1-Component-one?utm_source=chatgpt.com) \
\
Ejecute la siguiente celda para descargar los datos y cargarlos en un dataframe de pandas. La constante `DATA_PATH` determina la ubicación donde se almacenaran los datos.

In [ ]:
ds = load_dataset("tomas-gr/all-the-news-2-1-Component-one-sampled", split="train",cache_dir="../data")
df = ds.to_pandas()

## Lectura de Datos

In [ ]:
# Veamos las primeras filas del DataFrame
df.head()

In [ ]:
# Veamos información general del DataFrame
df.info()

# Parte 1: Cargado y Limpieza de Datos

## A. Exploración de Datos
Analice el contenido del DataFrame. Reporte si existen datos faltantes en algún campo, o cualquier otro problema de calidad de datos que encuentre. \
En particular, analice la cantidad de artículos por medio de prensa, y a partir de este punto trabaje con los **cinco medios con mayor cantidad de artículos**.

In [ ]:
# TODO: Analice datos faltantes por columna
df.isnull().sum()
col_not_full = df.columns[df.isnull().any()]
print("Columnas con datos faltantes:")
null_counts = df[col_not_full].isnull().sum()
for c in col_not_full:
    print(f"{c}: {null_counts[c]}/{len(df[c])} ({null_counts[c]/len(df[c])*100}%)")
#profile = ProfileReport(df, title="Data Profileing")
#profile.to_file("report.html")
#usamos reporte de la libreria ydata-profiling como baseline para comparar el analisiscon el nuestro

In [ ]:
#df_filtered = df[df['publication'].notnull() & ~df['publication'].str.isspace()]

#top_5_publications = df_filtered['publication'].value_counts().head(5)

#df_filtered = df_filtered[df_filtered['publication'].isin(top_5_publications.index)]
#top_5_publications

Eliminamos las tuplas de artículos que no tengan medio asociado (columna 'publication'), y nos quedamos con los artículos de los cinco medios con mayor cantidad de artículos.

In [ ]:
#date_only_pattern = r'^\d{4}-\d{2}-\d{2}$'
#datetime_pattern = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$'


#date_format_len = len(df_filtered[df_filtered['date'].str.strip().str.match(date_only_pattern)])
#datetime_format_len = len(df_filtered[df_filtered['date'].str.strip().str.match(datetime_pattern)])

#print(f'only date count: {date_format_len}')
#print(f'datetime count: {datetime_format_len}')
#print(f'sum: {date_format_len + datetime_format_len}')
#print(f'remaining date count: {len(df_filtered) - date_format_len - datetime_format_len}')

Se encuentran dos distintos formatos de fecha: una yyyy-mm-dd y yyyy-mm-dd hh:mm:ss (el segundo formato también incluye hora del día con presición de segundo). Esto puede llegar a ser un problema de exactitud sintáctica si el contexto requiere un formato específico para la fecha o incluso un problema de precisión si se requiere la hora de publicación de los artículos.

In [ ]:
# TODO: Analice la cantidad de artículos por medio de prensa

# Tome los 5 medios con más artículos
# top_5_publications = ...
# df_top_5 = ...

## B. Visualización temporal
Genere una gráfica que permita visualizar los artículos de los cinco medios a lo largo del tiempo, con alguna escala temporal adecuada. \
Comente si se identifican momentos de mayor actividad o patrones temporales en la cobertura.

In [ ]:
# TODO: Visualización de los artículos de cada medio a lo largo del tiempo
df_filtered=df[df['publication'].notnull()]
#normalized_date = pd.to_datetime(df['date'], errors='coerce')
#print((normalized_date==df['date']).sum())

#esto no  anda loparse a not a  timestaamp
# normalized_date2 = pd.to_datetime(df_filtered['date'], errors='coerce')
#print(f"fechas  corregidas { (normalized_date2==df_filtered['date']).sum() }")
#print(f"Enntradas  filtradas por  publicaacionn nulas: { len(df_filtered) - len(df) }") #  deberiamos filtrar por articulo y url? TODO
# df_filtered  =  df
# for x  in ["url","article",'publication']:
#    df_filtered=df[df_filtered[x].notnull()]
date_only_pattern = r'^\d{4}-\d{2}-\d{2}$'
datetime_pattern = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$'


date_format_len = len(df_filtered[df_filtered['date'].str.strip().str.match(date_only_pattern)])
datetime_format_len = len(df_filtered[df_filtered['date'].str.strip().str.match(datetime_pattern)])

print(f'only date count: {date_format_len}')
print(f'datetime count: {datetime_format_len}')
print(f'sum: {date_format_len + datetime_format_len}')
print(f'remaining date count: {len(df_filtered) - date_format_len - datetime_format_len}')

 
df_filtered['date'] 
# Preste especial atención al formato de la columna 'date', ya que puede contener diferentes formatos de fecha.


In [ ]:
def pparse_date(date_str):
    date_str = date_str.split(' ')
    
    return "".join(date_str[0].split("-"))  # Tomamos solo la parte de la fecha, ignorando la hora si está presente
df_filtered['date'] = df_filtered['date'].apply(pparse_date)
df_filtered['date']



In [ ]:

top_5_publications = df_filtered['publication'].value_counts().head(5)
df_filtered_top5=df_filtered[df_filtered['publication'].isin(top_5_publications.index)]

In [ ]:
#df_filtered_top5['date'] = pd.to_datetime(df_filtered_top5['date'].apply(pparse_date), errors='coerce')
#df_filtered_top5 = df_filtered_top5.dropna(subset=['date'])

# 2. Group by date and publication to get the counts (the Y axis)
# This creates a DataFrame where each column is a publication


plt.figure(figsize=(10, 6))
df_filtered_top5['date'] = pd.to_datetime(df_filtered_top5['date'], errors='coerce')
# Plotting the lists
colors =  sns.color_palette("tab10", n_colors=len(top_5_publications))
for i, publication in enumerate(top_5_publications.index):
    pub_data = df_filtered_top5[df_filtered_top5['publication'] == publication]
    pub_counts = pub_data.groupby('date').size().sort_index()
    pub_counts.index
    
    X = pub_counts.index
    Y = pub_counts.values
    plt.plot(X, Y, color=colors[i], label=publication)
plt.title('Comparaciont emporal en  conjunto')
plt.show()
for i, publication in enumerate(top_5_publications.index):
    plt.figure(figsize=(10, 6))
    pub_data = df_filtered_top5[df_filtered_top5['publication'] == publication]
    pub_counts = pub_data.groupby('date').size().sort_index()
    pub_counts.index
    
    X = pub_counts.index
    Y = pub_counts.values
    plt.plot(X, Y, color=colors[i], label=publication)
    plt.title('Comparaciont emporal individual')
    plt.show()

In [ ]:

#le aasaamos   un pasa bajo  para  visuaaliar mejor     
window_size = int(0.001*len(df_filtered_top5))
print(window_size)
plt.figure(figsize=(10, 6))
for i, publication in enumerate(top_5_publications.index):
    pub_data = df_filtered_top5[df_filtered_top5['publication'] == publication]
    pub_counts = pub_data.groupby('date').size().sort_index()
    pub_counts = pub_counts.rolling(window=window_size, win_type='gaussian').mean(std=3)
    
    X = pub_counts.index
    Y = pub_counts.values
    plt.plot(X, Y, color=colors[i], label=publication)
plt.title('Comparaciont emporal en  conjunto')
plt.show()
for i, publication in enumerate(top_5_publications.index):
    plt.figure(figsize=(10, 6))
    pub_data = df_filtered_top5[df_filtered_top5['publication'] == publication]
    pub_counts = pub_data.groupby('date').size().sort_index()
    pub_counts = pub_counts.rolling(window=window_size, win_type='gaussian').mean(std=3)
    
    X = pub_counts.index
    Y = pub_counts.values
    plt.plot(X, Y, color=colors[i], label=publication)
    plt.title(f'Comparaciont emporal {publication}')
    plt.show()

## C. Limpieza de texto y conteo de palabras
Se provee la función `clean_text(...)` que realiza parte de la normalización del texto. \
**Complete la función** agregando signos de puntuación faltantes y cualquier otra normalización que considere oportuna. \
Compruebe el resultado observando el contenido del DataFrame procesado. Comente todas las transformaciones que haya agregado y justifique.
    count.index = count.index.to_timestamp()
    ax.plot(count.index, count.values, label=publication)  

In [ ]:
# vamos aa filtrar los nombes de  los  publcitas    2 raoznees People  es  unaan  plaba normaal y the hills son  dols palaabarsa ->  creamos  un id unico  para cada publcista// soluciona poblemas en parte 2
pub  = df_filtered['publication'].unique()
pub_id = {p: f"pubid{p.replace(' ', '').lower()}" for p in pub}

In [ ]:
def preproces(S):
   
    if isinstance(S, str):    
        for p in pub:
            S = S.replace(p, pub_id[p])
    return S

In [ ]:
def not_a_letter(s):
    if not isinstance(s, str):
        return []
    return  [c for c in s if not c.isalpha()]
def get_not_letter(df, column):
    chars_df = df[column].apply(not_a_letter)
    all_chars = [char for sublist in chars_df for char in sublist] 
    return list(set(all_chars))
noalpha_chars = get_not_letter(df_filtered, 'article')
def countt_words(df, column_name):
    return df[column_name].apply(lambda x: str(x).split()).explode().value_counts()

In [ ]:


def clean_text(df, column_name):
    #noalpha_chars = get_not_letter(df, column_name)

   
    df2=df[column_name].apply(preproces)
    # Eliminar primeras palabras hasta el primer "\n"  xq??? 
    result = df2.str.replace(r"^[^\n]*\n", "", regex=True)

    # Convertir todo a minúsculas
    result = result.str.lower()
    print(noalpha_chars)
    # TODO: completar signos de puntuación faltantes
   # for punc in ["[", "\n", ",", ":", "?"]:
    for punc in noalpha_chars: #  esta aparte del codigfo ya estaba implementada lo emjor seria solo quedarme con lo  alpha   y  no  ahcer la lsita de  nlo  que  es no aalpha para nno  tenner  que hacer  un doble  fiulytrado
       result = result.str.replace(punc, " ")

    return result
clean_df=clean_text(df_filtered, 'article')

In [ ]:
# TODO: Aplique clean_text sobre la columna de texto elegida y cree una nueva columna "CleanText"
df_filtered['CleanText'] = clean_text(df_filtered, 'article')
df_filtered['CleanTextTitle'] = clean_text(df_filtered, 'title')




In [ ]:
cw=countt_words(df_filtered, 'CleanText')
for p  in  pub:
    if  pub_id[p].lower() in  cw.index:
        print(f"El medio {p} aparece en  {cw[pub_id[p].lower()]}  articulos")
#print(countt_words(df_filtered, 'CleanTextTitle').head(20))

## D. Elección de campos de texto
Discuta si conviene trabajar con:
- sólo el cuerpo del artículo,
- sólo el título,
- o una combinación de ambos.

Justifique brevemente su decisión.

*TODO: Escriba su análisis en el informe.*

## E. Pistas que identifican al medio de prensa
Analice si en el texto aparecen pistas que identifiquen de manera directa al medio de prensa (nombres del medio, URLs, firmas, nombres de secciones, plantillas repetidas, etc.). \
En caso de encontrarlas, comente si considera conveniente eliminarlas o reducir su impacto, y justifique su decisión.

In [ ]:
# TODO: Explore el texto buscando pistas que identifiquen directamente al medio de prensa
# Por ejemplo, busque nombres de medios, URLs, firmas, etc.
#df.columns
#for  pub  in df_filtered['publication'].unique():
#    df_filtered_i=df_filtered[df_filtered['publication']==pub]
#    counts = countt_words(df_filtered_i, 'CleanText')
#    if  pub.lower() in  counts.index:
#        print(f"El medio {pub} aparece en el texto con {counts[pub.lower()]} apariciones  en sus articulos")
#for  pub  in df_filtered['publication'].unique():
#    df_filtered_i=df_filtered[df_filtered['publication']==pub]
#    counts = countt_words(df_filtered_i, 'CleanTextTitle')
#    if  pub.lower() in  counts.index:
#        print(f"El medio {pub} aparece en el texto con {counts[pub.lower()]} apariciones  en sus titulos")

pubs = df_filtered['publication'].unique()
#pubs=  pubs[pubs != "People"]
#Eliminamos ppeople xq es una palabra comun no  solo un nombre propio 
plt.figure(figsize=(14, 12))
articulos_nombrar = np.zeros((len(pubs), len(pubs)))
titus_nombran    = np.zeros((len(pubs), len(pubs)))
for i, pubi in enumerate(pubs):
    df_filtered_i=df_filtered[df_filtered['publication']==pubi]
    counts = countt_words(df_filtered_i, 'CleanText')
    for  j,  pubj  in enumerate(pubs):
        if pubi==pubj:
            continue
        if  pub_id[pubj].lower() in  counts.index:
            articulos_nombrar[i,j]= counts[pub_id[pubj].lower()]
            #print(f"El medio {pubj} aparece en el texto con {counts[pubj.lower()]} apariciones  en sus articulos  de {pubi}")

sns.heatmap(
    articulos_nombrar,
    annot=True,      
    cmap='coolwarm',
    center=0,
    fmt='.0f',
    xticklabels=pubs,
    yticklabels=pubs
)
plt.show()


In [ ]:
plt.figure(figsize=(14, 12))
for i, pubi in enumerate(pubs):
    df_filtered_i=df_filtered[df_filtered['publication']==pubi]
    counts = countt_words(df_filtered_i, 'CleanTextTitle')
    for  j,  pubj  in enumerate(pubs):
        if pubi==pubj:
            continue
        if  pubj.lower() in  counts.index:
           titus_nombran[i,j]= counts[pubj.lower()]
           # print(f"El medio {pubj} aparece en el texto con {counts[pubj.lower()]} apariciones  en titulos de {pubi}")

c

sns.heatmap(
    titus_nombran,
    annot=True,      
    cmap='coolwarm',
    center=0,
     fmt='.0f',
    xticklabels=pubs,
    yticklabels=pubs
)

plt.show()

## F. Restricción por sección o período temporal
Evalúe si conviene restringir el análisis a artículos de una misma sección temática o de un período temporal acotado, con el objetivo de reducir el efecto del tema sobre una futura tarea de clasificación por medio. \
No es necesario implementar esta restricción, pero sí discutir sus posibles ventajas y desventajas.

*TODO: Escriba su análisis en el informe.*

# Parte 2: Conteo de Palabras y Visualizaciones

## A. Palabras más frecuentes por medio
Realice una visualización que permita comparar las palabras más frecuentes de cada uno de los cinco medios de prensa. \
Sin necesidad de implementarlo, proponga ideas para modificar esta visualización con el fin de encontrar diferencias entre secciones temáticas, fechas, o tipos de noticias.

In [ ]:
# TODO: Realice una visualización que permita comparar las palabras más frecuentes
# de cada uno de los cinco medios de prensa.
# - ¿Encuentra algún problema en los resultados?
from turtle import st
cw_top5 = {}
for pub in top_5_publications.index:
    df_pub = df_filtered[df_filtered['publication'] == pub]
    cw_top5[pub] = countt_words(df_pub, 'CleanText').sort_values(ascending=False)



for pub, counts in cw_top5.items():
        #TODO: pretify.#TODO: pretify.
    freq_dict = counts.head(50).to_dict()

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color="white"
    ).generate_from_frequencies(freq_dict)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.title(pub)
    plt.axis("off")
    plt.show()

word_complexity = 4
cw_top5 = {}
inverted_ids = {value: key for key, value in pub_id.items()} 
print(inverted_ids)     

for pub in top_5_publications.index:
    df_pub = df_filtered[df_filtered['publication'] == pub]
    cw_top5[pub] = countt_words(df_pub, 'CleanText').sort_values(ascending=False)
    cw_top5[pub].index = cw_top5[pub].index.to_series().apply(lambda x: x if x not in inverted_ids.keys() else inverted_ids[x])  # reemplazamos los id por los publishers originales 
for pub, counts in cw_top5.items():
    counts = counts[counts.index.to_series().apply(lambda x: len(x) > word_complexity)].head(50)
    freq_dict = counts.to_dict()
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color="white"
    ).generate_from_frequencies(freq_dict)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.title(f"Palabras en {pub}  mas  largas que {word_complexity}")
    plt.axis("off")
    plt.show()

    #me raaigo sot words de sklearn para  eliminar palabras comunes  
nltk.download('stopwords')
#stop_words = set(stopwords.words('english'))
stop_words = set (stopwords.words ('english')).union (ENGLISH_STOP_WORDS)
stop_words.update([pub_id[p].lower() for p in top_5_publications.index]) #  agregamos los id de los publicistas a la lista de stop words para  eliminar  las palabras que  son  los nombres de los publicistas y no  nos aportan informacion sobre el contenido de los articulos
for pub, counts in cw_top5.items():

    counts = counts[~counts.index.to_series().isin(stop_words)].head(50)
    freq_dict = counts.to_dict()
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color="white"
    ).generate_from_frequencies(freq_dict)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.title(f"Palabras en {pub}  sin stop words")
    plt.axis("off")
    plt.show()



## B. Medios con mayor cantidad de palabras
Corra el código que permite encontrar los medios con mayor cantidad de palabras. \
En caso de encontrar algún problema luego de realizar la visualización, comente a qué se debe y proponga formas de resolverlo.

In [ ]:

#aproximado afalta  agrregar el largo de las nombres  delos publicistas  -1 por cada vez que aparece su id    TODO 

for pub, counts in cw_top5.items():
    print(f"Palabras en {pub}: {counts.sum()} aprox")
    #cwtopp5 viene  de la partee  A  donde yaa corregimos los nombres de los publicistas
#    error= 0
    #for p in inverted_ids.keys():
    #    print(p)
    #    print( counts.index)
    #    if p in counts.index:
    #        print(f"El id {p} aparece {counts[p]} veces en {pub}")
    #        error += len(inverted_ids[p].split(" ")) - 1  # restamos 1 por cada aparición del id, ya que el id reemplaza al nombre completo del publicista
    #print(f"Palabras en {pub}: {counts.sum()+error} error  corregido")
    error=0
    for p in cw_top5.keys():
        error = counts.get(p, 0) * (len(p.split(" ")) - 1)  # restamos 1 por cada aparición del id, xqya esta conttada
    print(f"Palabras en {pub}: {counts.sum()+error} error  corregido")   
    

## C. Matriz de menciones entre medios
Construya una matriz de 5×5, donde cada fila y columna corresponden a un medio de prensa, y la entrada (i,j) contiene la cantidad de veces que el medio *i* menciona al medio *j*. \
\
**Opcional:** genere un grafo dirigido con esa matriz de adyacencia para visualizar las menciones. Puede ser útil la biblioteca `networkx`.

In [ ]:
#df_filtered_top5['CleanText'] = clean_text(df_filtered_top5, 'article')
#df_filtered_top5['CleanTextTitle'] = clean_text(df_filtered_top5, 'title')

pubs = df_filtered_top5['publication'].unique()
#pubs=  pubs[pubs != "People"]
#Eliminamos ppeople xq es una palabra comun no  solo un nombre propio 
plt.figure(figsize=(14, 12))
articulos_nombrar2   = np.zeros((len(pubs), len(pubs)))

for i, pubi in enumerate(pubs):
    df_filtered_i=df_filtered_top5[df_filtered_top5['publication']==pubi]
    counts = countt_words(df_filtered_i, 'CleanText')
    for  j,  pubj  in enumerate(pubs):
        #if pubi==pubj:
        #    continue

        if  pubj.lower() in  counts.index:
            articulos_nombrar2[i,j]= counts[pubj.lower()]
            #print(f"El medio {pubj} aparece en el texto con {counts[pubj.lower()]} apariciones  en sus articulos  de {pubi}")

sns.heatmap(
    articulos_nombrar2,
    annot=True,      
    cmap='coolwarm',
    center=0,
    fmt='.0f',
    xticklabels=pubs,
    yticklabels=pubs
)
plt.show()

plt.figure(figsize=(14, 12))

titus_nombran2    = np.zeros((len(pubs), len(pubs)))
for i, pubi in enumerate(pubs):
    df_filtered_i=df_filtered_top5[df_filtered_top5['publication']==pubi]
    counts = countt_words(df_filtered_i, 'CleanTextTitle')
    for  j,  pubj  in enumerate(pubs):
        #if pubi==pubj:
        #    continue

        if  pubj.lower() in  counts.index:
            titus_nombran2[i,j]= counts[pubj.lower()]
            #print(f"El medio {pubj} aparece en el texto con {counts[pubj.lower()]} apariciones  en sus articulos  de {pubi}")

sns.heatmap(
    titus_nombran2,
    annot=True,      
    cmap='coolwarm',
    center=0,
    fmt='.0f',
    xticklabels=pubs,
    yticklabels=pubs
)
plt.show()

In [ ]:
# Opcional: Genere un grafo dirigido con la matriz de adyacencia para visualizar las menciones.
# Puede ser útil la biblioteca networkx.
#https://networkx.org/documentation/stable/reference/drawing.html
def make_grapph(mat):
    G = nx.DiGraph()
    for pub in pubs:
        G.add_node(pub)

    # aristas
    for i, origen in enumerate(pubs):
        for j, destino in enumerate(pubs):

            peso = mat[i, j]

            # solo conexiones con menciones
            if peso > 0:
                G.add_edge(origen, destino, weight=peso)

    plt.figure(figsize=(16, 12))

    pos = nx.spring_layout(G, seed=42)

    nx.draw_networkx_nodes(
        G,
        pos,
        node_size=2500
    )

    nx.draw_networkx_labels(
        G,
        pos,
        font_size=10
    )

    nx.draw_networkx_edges(
        G,
        pos,
        arrows=True,
        arrowstyle='->',
        arrowsize=20
    )
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(
        G,
        pos,
        edge_labels=edge_labels,
        font_size=8
    )

    plt.axis('off')
    plt.show()
make_grapph(articulos_nombrar2)
make_grapph(titus_nombran2)

## D. Preguntas propuestas
Proponga al menos tres preguntas que se podrían intentar responder a partir de estos datos, y mencione posibles caminos para responderlas, sin implementar nada.

*TODO: Escriba sus preguntas y posibles caminos en el informe.*